# Notebook 1A - ETL Run Once (Standalone e Didatico)

Objetivo:
- Executar ETL completo sem dependencia de `src/etl_core.py`.
- Processar arquivos em `data/pending` e carregar no PostgreSQL.
- Demonstrar bloco a bloco como no estilo didatico do professor.

Regras de sobrescrita:
- Movimentacoes: chave de negocio `id_movimentacao + id_produto_servico + id_pessoa`.
- Produtos: chave `id_produto_servico`.


## Bloco 1 - Configuracao de ambiente e conexao


In [1]:
import os
import csv
import json
import hashlib
from pathlib import Path
from datetime import datetime
from decimal import Decimal

from sqlalchemy import create_engine, text

INPUT_DIR = Path(os.getenv("INPUT_DIR", "/workspace/data/pending"))
PROCESSED_DIR = Path(os.getenv("PROCESSED_DIR", "/workspace/data/processed"))
FAILED_DIR = Path(os.getenv("FAILED_DIR", "/workspace/data/failed"))

DB_USER = os.getenv("POSTGRES_USER", "fs_user")
DB_PASS = os.getenv("POSTGRES_PASSWORD", "fs_pass")
DB_HOST = os.getenv("POSTGRES_HOST", "postgres")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB", "fs_mix")

conn_str = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_str, future=True)

INPUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FAILED_DIR.mkdir(parents=True, exist_ok=True)

print("INPUT_DIR:", INPUT_DIR)
print("Conexao:", f"{DB_HOST}:{DB_PORT}/{DB_NAME}")


INPUT_DIR: /workspace/data/pending
Conexao: postgres:5432/fs_mix


## Bloco 2 - Funcoes


In [2]:
# Le arquivo texto tentando encodings comuns.
def ler_texto(path: Path) -> str:
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            return path.read_text(encoding=enc)
        except Exception:
            continue
    return path.read_text(encoding="utf-8", errors="replace")


# Converte JSON/JSONL em lista de registros.
def ler_registros_json(path: Path):
    raw = ler_texto(path).strip()
    if not raw:
        return []
    try:
        obj = json.loads(raw)
        if isinstance(obj, list):
            return [r for r in obj if isinstance(r, dict)]
        if isinstance(obj, dict):
            return [obj]
    except Exception:
        pass

    out = []
    for line in raw.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            if isinstance(obj, dict):
                out.append(obj)
        except Exception:
            continue
    return out


# Converte CSV em lista de dicionarios normalizados.
def ler_registros_csv(path: Path):
    rows = []
    for row in csv.DictReader(ler_texto(path).splitlines()):
        rows.append({(k or "").strip(): (v.strip() if isinstance(v, str) else v) for k, v in row.items()})
    return rows


# Identifica tipo de arquivo para a trilha de ETL.
def detectar_tipo_arquivo(path: Path) -> str:
    n = path.name.lower()
    if n.startswith("movimentacoes") and path.suffix.lower() == ".json":
        return "movimentacoes_json"
    if n.startswith("produtos_servicos") and path.suffix.lower() == ".csv":
        return "produtos_servicos_csv"
    return "unsupported"


# Calcula hash do arquivo para controle de processamento.
def calcular_hash_arquivo(path: Path) -> str:
    d = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            d.update(chunk)
    return d.hexdigest()


# Gera hash estavel por registro para deduplicacao tecnica.
def gerar_hash_registro(record) -> str:
    payload = json.dumps(record, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


# Converte valor para Decimal com tolerancia a formato.
def para_decimal(value):
    if value in (None, "", "null"):
        return None
    try:
        return Decimal(str(value).replace(",", "."))
    except Exception:
        return None


# Converte representacoes textuais em booleano.
def para_booleano(value):
    if value is None or value == "":
        return None
    if isinstance(value, bool):
        return value
    t = str(value).strip().lower()
    if t in {"true", "1", "sim", "s", "yes", "y"}:
        return True
    if t in {"false", "0", "nao", "não", "n", "no"}:
        return False
    return None


# Converte texto de data para objeto date.
def para_data(value):
    if value in (None, "", "null"):
        return None
    value = str(value).strip()
    for fmt in ("%Y-%m-%d", "%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S"):
        try:
            return datetime.strptime(value[:19], fmt).date()
        except Exception:
            continue
    return None


## Bloco 3 - Funcoes de carga e upsert


In [3]:
# Registra status do arquivo no file_registry.
def registrar_arquivo(conn, file_name, file_hash, file_type, status, rows_loaded=0, error_message=""):
    conn.execute(
        text(
            """
            INSERT INTO bronze.file_registry (file_name, file_hash, file_type, status, rows_loaded, error_message, processed_at)
            VALUES (:file_name, :file_hash, :file_type, :status, :rows_loaded, :error_message, NOW())
            ON CONFLICT (file_name) DO UPDATE SET
                file_hash = EXCLUDED.file_hash,
                file_type = EXCLUDED.file_type,
                status = EXCLUDED.status,
                rows_loaded = EXCLUDED.rows_loaded,
                error_message = EXCLUDED.error_message,
                processed_at = NOW()
            """
        ),
        {
            "file_name": file_name,
            "file_hash": file_hash,
            "file_type": file_type,
            "status": status,
            "rows_loaded": rows_loaded,
            "error_message": error_message,
        },
    )


# Move arquivo para pasta de destino evitando sobrescrita.
def mover_arquivo(path: Path, target_dir: Path):
    dest = target_dir / path.name
    if dest.exists():
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        dest = target_dir / f"{path.stem}_{stamp}{path.suffix}"
    path.replace(dest)


# Carrega movimentacoes no bronze e upsert na fato silver.
def carregar_movimentacoes(conn, source_file, records):
    conn.execute(text("DELETE FROM bronze.movimentacoes_raw WHERE source_file = :f"), {"f": source_file})

    loaded = 0
    skipped_invalid_key = 0

    for idx, rec in enumerate(records, start=1):
        rec_hash = gerar_hash_registro(rec)

        conn.execute(
            text(
                """
                INSERT INTO bronze.movimentacoes_raw (source_file, source_row, record_hash, payload)
                VALUES (:source_file, :source_row, :record_hash, CAST(:payload AS JSONB))
                ON CONFLICT (record_hash) DO NOTHING
                """
            ),
            {
                "source_file": source_file,
                "source_row": idx,
                "record_hash": rec_hash,
                "payload": json.dumps(rec, ensure_ascii=False, default=str),
            },
        )

        id_mov = str(rec.get("id_movimentacao") or "").strip()
        id_prod = str(rec.get("id_produto_servico") or "").strip()
        id_pessoa = str(rec.get("id_pessoa") or "").strip()

        if not id_mov or not id_prod or not id_pessoa:
            skipped_invalid_key += 1
            continue

        qtd_venda = para_decimal(rec.get("qtd_venda"))
        valor_unitario = para_decimal(rec.get("valor_unitario"))
        desconto_dig = para_decimal(rec.get("valor_desconto_digitado")) or Decimal("0")
        desconto_prop = para_decimal(rec.get("valor_desconto_proporcional")) or Decimal("0")
        frete = para_decimal(rec.get("valor_frete_item")) or Decimal("0")

        bruto = (qtd_venda or Decimal("0")) * (valor_unitario or Decimal("0"))
        liquido = bruto - desconto_dig - desconto_prop + frete

        conn.execute(
            text(
                """
                INSERT INTO silver.fato_movimentacoes (
                    record_hash, source_file, source_row, data_emissao,
                    id_movimentacao, id_produto_servico, id_produto_servico_empresa,
                    cd_cfop, descricao_cfop, cd_modelo, descricao_modelo,
                    cd_modelo_fiscal, cd_situacao, descricao_situacao,
                    direcao_estoque, tipo_transacao, id_pessoa,
                    status_item_cancelado, qtd_item_movimentacao, qtd_venda,
                    valor_unitario, valor_desconto_digitado, valor_desconto_proporcional,
                    valor_frete_item, valor_total_bruto, valor_total_liquido
                ) VALUES (
                    :record_hash, :source_file, :source_row, :data_emissao,
                    :id_movimentacao, :id_produto_servico, :id_produto_servico_empresa,
                    :cd_cfop, :descricao_cfop, :cd_modelo, :descricao_modelo,
                    :cd_modelo_fiscal, :cd_situacao, :descricao_situacao,
                    :direcao_estoque, :tipo_transacao, :id_pessoa,
                    :status_item_cancelado, :qtd_item_movimentacao, :qtd_venda,
                    :valor_unitario, :valor_desconto_digitado, :valor_desconto_proporcional,
                    :valor_frete_item, :valor_total_bruto, :valor_total_liquido
                )
                ON CONFLICT (id_movimentacao, id_produto_servico, id_pessoa) DO UPDATE SET
                    record_hash = EXCLUDED.record_hash,
                    source_file = EXCLUDED.source_file,
                    source_row = EXCLUDED.source_row,
                    data_emissao = EXCLUDED.data_emissao,
                    id_produto_servico_empresa = EXCLUDED.id_produto_servico_empresa,
                    cd_cfop = EXCLUDED.cd_cfop,
                    descricao_cfop = EXCLUDED.descricao_cfop,
                    cd_modelo = EXCLUDED.cd_modelo,
                    descricao_modelo = EXCLUDED.descricao_modelo,
                    cd_modelo_fiscal = EXCLUDED.cd_modelo_fiscal,
                    cd_situacao = EXCLUDED.cd_situacao,
                    descricao_situacao = EXCLUDED.descricao_situacao,
                    direcao_estoque = EXCLUDED.direcao_estoque,
                    tipo_transacao = EXCLUDED.tipo_transacao,
                    status_item_cancelado = EXCLUDED.status_item_cancelado,
                    qtd_item_movimentacao = EXCLUDED.qtd_item_movimentacao,
                    qtd_venda = EXCLUDED.qtd_venda,
                    valor_unitario = EXCLUDED.valor_unitario,
                    valor_desconto_digitado = EXCLUDED.valor_desconto_digitado,
                    valor_desconto_proporcional = EXCLUDED.valor_desconto_proporcional,
                    valor_frete_item = EXCLUDED.valor_frete_item,
                    valor_total_bruto = EXCLUDED.valor_total_bruto,
                    valor_total_liquido = EXCLUDED.valor_total_liquido,
                    ingestion_ts = NOW()
                """
            ),
            {
                "record_hash": rec_hash,
                "source_file": source_file,
                "source_row": idx,
                "data_emissao": para_data(rec.get("data_emissao")),
                "id_movimentacao": id_mov,
                "id_produto_servico": id_prod,
                "id_produto_servico_empresa": str(rec.get("id_produto_servico_empresa") or "") or None,
                "cd_cfop": str(rec.get("cd_cfop") or "") or None,
                "descricao_cfop": str(rec.get("descricao_cfop") or "") or None,
                "cd_modelo": str(rec.get("cd_modelo") or "") or None,
                "descricao_modelo": str(rec.get("descricao_modelo") or "") or None,
                "cd_modelo_fiscal": str(rec.get("cd_modelo_fiscal") or "") or None,
                "cd_situacao": str(rec.get("cd_situacao") or "") or None,
                "descricao_situacao": str(rec.get("descricao_situacao") or "") or None,
                "direcao_estoque": str(rec.get("direcao_estoque") or "") or None,
                "tipo_transacao": str(rec.get("tipo_transacao") or "") or None,
                "id_pessoa": id_pessoa,
                "status_item_cancelado": para_booleano(rec.get("status_item_cancelado")),
                "qtd_item_movimentacao": para_decimal(rec.get("qtd_item_movimentacao")),
                "qtd_venda": qtd_venda,
                "valor_unitario": valor_unitario,
                "valor_desconto_digitado": para_decimal(rec.get("valor_desconto_digitado")),
                "valor_desconto_proporcional": para_decimal(rec.get("valor_desconto_proporcional")),
                "valor_frete_item": para_decimal(rec.get("valor_frete_item")),
                "valor_total_bruto": bruto,
                "valor_total_liquido": liquido,
            },
        )
        loaded += 1

    return loaded, skipped_invalid_key


# Carrega produtos no bronze e upsert na dimensao silver.
def carregar_produtos(conn, source_file, records):
    conn.execute(text("DELETE FROM bronze.produtos_servicos_raw WHERE source_file = :f"), {"f": source_file})

    loaded = 0
    for idx, rec in enumerate(records, start=1):
        rec_hash = gerar_hash_registro(rec)

        conn.execute(
            text(
                """
                INSERT INTO bronze.produtos_servicos_raw (source_file, source_row, record_hash, payload)
                VALUES (:source_file, :source_row, :record_hash, CAST(:payload AS JSONB))
                ON CONFLICT (record_hash) DO NOTHING
                """
            ),
            {
                "source_file": source_file,
                "source_row": idx,
                "record_hash": rec_hash,
                "payload": json.dumps(rec, ensure_ascii=False, default=str),
            },
        )

        id_produto = str(rec.get("id_produto_servico") or "").strip()
        if not id_produto:
            continue

        conn.execute(
            text(
                """
                INSERT INTO silver.dim_produtos_servicos (
                    id_produto_servico, cd_produto_servico, descricao, status_produto_servico,
                    pesavel, vendavel, percentual_cashback, unidade_sigla, tipo_item_descricao,
                    sub_grupo_referencia, codigo_barras, codigo_barras_tributavel,
                    id_produto_servico_empresa_referencia, id_empresa_referencia,
                    id_estoque_referencia, id_preco_referencia,
                    margem_lucro_aplicada_referencia, limite_desconto_referencia,
                    percentual_comissao_referencia, source_file
                ) VALUES (
                    :id_produto_servico, :cd_produto_servico, :descricao, :status_produto_servico,
                    :pesavel, :vendavel, :percentual_cashback, :unidade_sigla, :tipo_item_descricao,
                    :sub_grupo_referencia, :codigo_barras, :codigo_barras_tributavel,
                    :id_produto_servico_empresa_referencia, :id_empresa_referencia,
                    :id_estoque_referencia, :id_preco_referencia,
                    :margem_lucro_aplicada_referencia, :limite_desconto_referencia,
                    :percentual_comissao_referencia, :source_file
                )
                ON CONFLICT (id_produto_servico) DO UPDATE SET
                    cd_produto_servico = EXCLUDED.cd_produto_servico,
                    descricao = EXCLUDED.descricao,
                    status_produto_servico = EXCLUDED.status_produto_servico,
                    pesavel = EXCLUDED.pesavel,
                    vendavel = EXCLUDED.vendavel,
                    percentual_cashback = EXCLUDED.percentual_cashback,
                    unidade_sigla = EXCLUDED.unidade_sigla,
                    tipo_item_descricao = EXCLUDED.tipo_item_descricao,
                    sub_grupo_referencia = EXCLUDED.sub_grupo_referencia,
                    codigo_barras = EXCLUDED.codigo_barras,
                    codigo_barras_tributavel = EXCLUDED.codigo_barras_tributavel,
                    id_produto_servico_empresa_referencia = EXCLUDED.id_produto_servico_empresa_referencia,
                    id_empresa_referencia = EXCLUDED.id_empresa_referencia,
                    id_estoque_referencia = EXCLUDED.id_estoque_referencia,
                    id_preco_referencia = EXCLUDED.id_preco_referencia,
                    margem_lucro_aplicada_referencia = EXCLUDED.margem_lucro_aplicada_referencia,
                    limite_desconto_referencia = EXCLUDED.limite_desconto_referencia,
                    percentual_comissao_referencia = EXCLUDED.percentual_comissao_referencia,
                    source_file = EXCLUDED.source_file,
                    ingestion_ts = NOW()
                """
            ),
            {
                "id_produto_servico": id_produto,
                "cd_produto_servico": str(rec.get("cd_produto_servico") or "") or None,
                "descricao": str(rec.get("descricao") or "") or None,
                "status_produto_servico": str(rec.get("status_produto_servico") or "") or None,
                "pesavel": para_booleano(rec.get("pesavel")),
                "vendavel": para_booleano(rec.get("vendavel")),
                "percentual_cashback": para_decimal(rec.get("percentual_cashback")),
                "unidade_sigla": str(rec.get("unidade_sigla") or "") or None,
                "tipo_item_descricao": str(rec.get("tipo_item_descricao") or "") or None,
                "sub_grupo_referencia": str(rec.get("sub_grupo_referencia") or "") or None,
                "codigo_barras": str(rec.get("codigo_barras") or "") or None,
                "codigo_barras_tributavel": str(rec.get("codigo_barras_tributavel") or "") or None,
                "id_produto_servico_empresa_referencia": str(rec.get("id_produto_servico_empresa_referencia") or "") or None,
                "id_empresa_referencia": str(rec.get("id_empresa_referencia") or "") or None,
                "id_estoque_referencia": str(rec.get("id_estoque_referencia") or "") or None,
                "id_preco_referencia": str(rec.get("id_preco_referencia") or "") or None,
                "margem_lucro_aplicada_referencia": para_decimal(rec.get("margem_lucro_aplicada_referencia")),
                "limite_desconto_referencia": para_decimal(rec.get("limite_desconto_referencia")),
                "percentual_comissao_referencia": para_decimal(rec.get("percentual_comissao_referencia")),
                "source_file": source_file,
            },
        )

        loaded += 1

    return loaded


## Bloco 4 - Execucao executar_uma_vez (arquivo por arquivo)


In [4]:
files = sorted([p for p in INPUT_DIR.iterdir() if p.is_file()])
report = []

for p in files:
    ftype = detectar_tipo_arquivo(p)

    if ftype == "unsupported":
        file_hash = calcular_hash_arquivo(p)
        msg = "Padrao de nome de arquivo nao suportado. Esperado: movimentacoes*.json ou produtos_servicos*.csv."
        with engine.begin() as conn:
            registrar_arquivo(conn, p.name, file_hash, ftype, "FAILED", 0, msg)
        mover_arquivo(p, FAILED_DIR)
        report.append((p.name, ftype, "FAILED", 0, 0))
        continue

    file_hash = calcular_hash_arquivo(p)

    with engine.begin() as conn:
        registrar_arquivo(conn, p.name, file_hash, ftype, "RUNNING", 0, "")

    try:
        records = ler_registros_json(p) if ftype == "movimentacoes_json" else ler_registros_csv(p)

        with engine.begin() as conn:
            if ftype == "movimentacoes_json":
                loaded, skipped_invalid_key = carregar_movimentacoes(conn, p.name, records)
            else:
                loaded = carregar_produtos(conn, p.name, records)
                skipped_invalid_key = 0

            registrar_arquivo(conn, p.name, file_hash, ftype, "SUCCESS", loaded, "")

        mover_arquivo(p, PROCESSED_DIR)
        report.append((p.name, ftype, "SUCCESS", loaded, skipped_invalid_key))

    except Exception as exc:
        with engine.begin() as conn:
            registrar_arquivo(conn, p.name, file_hash, ftype, "FAILED", 0, str(exc)[:2000])
        mover_arquivo(p, FAILED_DIR)
        report.append((p.name, ftype, "FAILED", 0, 0))

for row in report:
    print(f"{row[0]:40} | {row[1]:24} | {row[2]:8} | carregados: {row[3]:4} | chave invalida: {row[4]:3}")


movimentacoes_20260506.json              | movimentacoes_json       | SUCCESS  | carregados:   70 | chave invalida:   0
movimentacoes_20260507.json              | movimentacoes_json       | SUCCESS  | carregados:   56 | chave invalida:   0
movimentacoes_20260508.json              | movimentacoes_json       | SUCCESS  | carregados:   58 | chave invalida:   0
produtos_servicos_20260512.csv           | produtos_servicos_csv    | SUCCESS  | carregados: 1389 | chave invalida:   0


## Bloco 5 - Validacao rapida no banco


In [5]:
with engine.begin() as conn:
    rows = conn.execute(text("""
        SELECT 'bronze.file_registry' AS tabela, COUNT(*) AS total FROM bronze.file_registry
        UNION ALL
        SELECT 'bronze.movimentacoes_raw', COUNT(*) FROM bronze.movimentacoes_raw
        UNION ALL
        SELECT 'bronze.produtos_servicos_raw', COUNT(*) FROM bronze.produtos_servicos_raw
        UNION ALL
        SELECT 'silver.fato_movimentacoes', COUNT(*) FROM silver.fato_movimentacoes
        UNION ALL
        SELECT 'silver.dim_produtos_servicos', COUNT(*) FROM silver.dim_produtos_servicos
        ORDER BY tabela
    """)).fetchall()

for r in rows:
    print(dict(r._mapping))


{'tabela': 'bronze.file_registry', 'total': 14}
{'tabela': 'bronze.movimentacoes_raw', 'total': 5337}
{'tabela': 'bronze.produtos_servicos_raw', 'total': 1389}
{'tabela': 'silver.dim_produtos_servicos', 'total': 1389}
{'tabela': 'silver.fato_movimentacoes', 'total': 4947}


## Desafios e solucoes

1. Diferencas de encoding: leitura tolerante com multiplos encodings.
2. Reprocessamento: upsert por chave de negocio em movimentacoes.
3. Auditabilidade: `file_registry` com status, linhas e horario.
4. Operacao: `pending` (entrada), `processed` (sucesso), `failed` (erro).

Observacao:
- O Notebook 1B e a evolucao operacional em loop continuo usando `src/etl_core.py`.
